# CAD ELT walkthrough

Portfolio write-up of the JPL Close Approach Data pipeline: **API → Snowflake Bronze → Dynamic Tables (Silver/Gold) → Streamlit**, scheduled with **GitHub Actions**.

This notebook is meant to be **read on GitHub**. It does not connect to Snowflake.

Repo layout: `src/` (Python), `sql/` (Snowflake), `.github/workflows/cad-elt.yml` (cron).

## What CAD is

NASA/JPL [Close Approach Data](https://ssd-api.jpl.nasa.gov/doc/cad.html) lists predicted times when near-Earth asteroids (and some comets) pass Earth: designation, time, miss distance, relative speed.

It is an **ephemeris** (calculated flybys), not a live radar feed. A weekly extract is enough.

This project pulls `GET https://ssd-api.jpl.nasa.gov/cad.api` with a date window, `dist-max=0.05` AU, and `nea=true`, then keeps one JSON document in Bronze (`extracted_at` + `payload VARIANT`).

## End-to-end flow

```
JPL CAD API
    │  extract_cad_elt.py          (GitHub runner, Mondays 06:00 UTC)
    ▼
JSON on the runner (not committed)
    │  load_cad_snowflake.py       warehouse: SNOWFLAKE_LEARNING_WH
    ▼
ELT_DEV.BRONZE.CAD_RAW             one row, full JSON
    │  refresh_cad_dts.py          ALTER DYNAMIC TABLE … REFRESH
    ▼
SILVER.CAD  →  GOLD.CAD_FLYBYS / CAD_KPI / CAD_BY_DAY
    │  src/app.py                  warehouse: DASHBOARD_WH
    ▼
Streamlit (local): KPIs, flyby table, approaches per day
```

ELT, not ETL: Python does extract + load. Transform stays in Snowflake (Dynamic Tables + `LATERAL FLATTEN`).

## 1. Extract

`src/extract_cad_elt.py` — `requests.get` with a User-Agent, query params, JSON dump to `data/raw/cad/cad_<UTC stamp>.json`.

The raw folder is gitignored. On GitHub Actions the file lives only for the length of the job, then the runner is gone.

## 2. Load (Bronze)

`src/load_cad_snowflake.py` connects with `SNOWFLAKE_WAREHOUSE` (load WH, not the dashboard WH).

- `CREATE TABLE IF NOT EXISTS CAD_RAW (extracted_at STRING, payload VARIANT)`
- `DELETE` then `INSERT … SELECT %s, PARSE_JSON(%s)` so Bronze stays **one row** (latest snapshot)

Proof that a run happened is `extracted_at` (filename stem), not a growing row count. Example after a Monday cron: `cad_20261019T120645Z`.

## 3. Transform (Dynamic Tables)

DDL: `sql/cad_dynamic_tables.sql`. Learning path (CTAS, then DROP, then DTs): `sql/cad_archive.sql` — do not run that file as a whole.

| Object | Layer | Lag | Role |
|---|---|---|---|
| `SILVER.CAD` | flatten `payload:data` | `TARGET_LAG = '3 days'` | typed flybys (AU, km/s) |
| `GOLD.CAD_FLYBYS` | from Silver | `DOWNSTREAM` | km + closeness bucket |
| `GOLD.CAD_KPI` | from Silver | `DOWNSTREAM` | counts / min distance / max speed |
| `GOLD.CAD_BY_DAY` | from Silver | `DOWNSTREAM` | approaches per calendar day |

All DTs use `SNOWFLAKE_LEARNING_WH`.

**Why 3-day lag if Actions already run Mondays?** Transform should live in Snowflake. Lag is a safety net if the Action fails. `ALTER DYNAMIC TABLE … REFRESH` in the Action is what makes Gold match Bronze **on Monday**, without waiting up to three days.

`src/refresh_cad_dts.py` runs those `ALTER`s after load (Silver first, then Gold).

## 4. Orchestration (GitHub Actions)

`.github/workflows/cad-elt.yml`

- `schedule: cron 0 6 * * 1` (Monday 06:00 UTC)
- `workflow_dispatch` (manual)
- steps: checkout → Python 3.12 → `pip install -r requirements.txt` → extract → load → refresh

Secrets (same keys as `.env`, **not** the dashboard warehouse): account, user, password, load warehouse, database, schema, role.

A **new** run after a YAML change must be **Run workflow**, not Re-run of an old job (that uses the old commit).

Where to see the last run: GitHub **Actions** tab, and `SELECT extracted_at FROM ELT_DEV.BRONZE.CAD_RAW`.

## 5. Two warehouses

| Env var | Typical name | Used by |
|---|---|---|
| `SNOWFLAKE_WAREHOUSE` | `SNOWFLAKE_LEARNING_WH` | load + DT refresh |
| `SNOWFLAKE_DASHBOARD_WAREHOUSE` | `DASHBOARD_WH` | Streamlit `SELECT`s only |

`DASHBOARD_WH`: XSMALL, `AUTO_SUSPEND = 60`, `AUTO_RESUME = TRUE`. A dashboard rerun should not wake the load warehouse.

Credits only while a warehouse is **running**. First Streamlit open after suspend can take ~1 minute (resume).

## 6. Dashboard

`src/app.py` — `python -m streamlit run src/app.py`.

Reads `GOLD.CAD_KPI`, `CAD_FLYBYS` (closest 100), `CAD_BY_DAY`. No extract/load in the app.

Streamlit is **not** scheduled. It only reads Gold.

## Screenshots

Drop files into `docs/screenshots/` with these names (crop account URLs, email, credit balance).

### GitHub Actions — green weekly run

![GitHub Actions run](screenshots/01_github_actions.png)

### GitHub Actions — job steps (extract, load, refresh)

![Action steps](screenshots/02_action_steps.png)

### Snowflake Bronze — `extracted_at`

![Bronze extracted_at](screenshots/03_bronze_extracted_at.png)

### Snowflake Gold KPI

![Gold KPI](screenshots/04_gold_kpi.png)

### Snowflake Dynamic Tables (Gold)

![SHOW DYNAMIC TABLES](screenshots/05_dynamic_tables.png)

### Warehouses (names, size, auto-suspend only)

![SHOW WAREHOUSES](screenshots/06_warehouses.png)

### Streamlit dashboard

![Streamlit](screenshots/07_streamlit.png)